# CAR-bench Agent Alignment: Supervised Fine-Tuning (SFT) Pipeline (Qwen 3.5 4B)

This notebook contains a complete training pipeline to perform Supervised Fine-Tuning (SFT) on a Large Language Model (LLM) for the CAR-bench voice assistant competition.

## Contents
1. Environment Setup: Install packages and verify environment.
2. CAR-bench Evaluation Strategy: Understand deterministic vs LLM-prompted metrics.
3. Data Preparation: Load datasets and parse trajectories.
4. Formatting and Tokenization: Apply chat templates for SFT format.
5. Model Loading: Initialize Qwen 3.5 4B weights and configure PEFT/LoRA adapters.
6. Training Configuration: Set training hyperparameters and execute SFT with evaluation and best checkpoint saving.
7. Visualization: Plot train vs validation losses.
8. Saving and Merging Model Weights: Save adapters and merge to 16-bit precision weights.
9. Inference Server Deployment: Launch local server for evaluation.

## 1. Environment Setup

In [ ]:
# Running in local offline workstation environment
IN_COLAB = False
print("Running on local workstation. Repositories and paths are assumed local.")

In [ ]:
# Ensure you have installed these packages in your environment (e.g. conda environment):
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# pip install "transformers>=5.0.0" peft trl datasets accelerate bitsandbytes matplotlib seaborn jupyterlab ipykernel
print("Please verify packages are pre-installed in your virtual environment.")

In [ ]:
import os
import sys
import json
import torch
import random

# Establish seeds for training reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. CAR-bench Evaluation Strategy & Metrics

Based on the official CAR-bench evaluation design, the benchmark is significantly more deterministic than it may initially appear. The evaluation is split into:

### 1. Exact / Deterministic Evaluation
* **State-based reward**: Replays ground truth actions to produce expected state hashes, then compares them (`==`) with the agent's state hashes (`base.py:81`).
* **Tool subset reward**: Checks whether the agent called every tool that the ground truth called (`gt_action_names.issubset(performed_action_names)`) (`reward_calculators.py:123`).
* **Tool execution errors**: Checks whether the runtime error list is empty (accumulated during tool execution).
* **Policy (AUT)**: ~200 lines of deterministic checks (tool call names, argument values, vehicle state) (`policy_evaluator.py:116-325`).
* **Success threshold**: Exact float comparison (`1 - 1e-6 <= reward <= 1 + 1e-6`).

### 2. LLM-Prompted Evaluation
* **Policy (LLM)**: ~20 domain-specific policies are checked by sending `(policy, trajectory)` to an LLM judge asking for `{policy_followed: bool, reasoning: str}` (`policy_evaluator.py:88-114`).
* **End conversation**: The user simulator (LLM) decides each turn (`CONTINUE`, `STOP`, `HALLUCINATION_ERROR`, `DISAMBIGUATION_ERROR`, etc.) acting as a quality gate.

> [!NOTE]
> The LLM judge is a supplement, not the core evaluator. Our SFT training focuses on mimicking these precise deterministic rules and tool calling behaviors.

## 3. Data Preparation

In [ ]:
PERSISTENT_DIR = "./outputs"
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target checkpoints directory configured locally: {PERSISTENT_DIR}")

In [ ]:
import gc

def clear_gpu_memory():
    """Purges system garbage collection and releases cached PyTorch CUDA memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Released GPU cache memory.")

clear_gpu_memory()

In [ ]:
from datasets import load_dataset

# Tải dataset trực tiếp từ Hugging Face
print("Loading dataset from Hugging Face: upwitu/trash_draft_am...")
try:
    dataset = load_dataset("upwitu/trash_draft_am", split="train")
except Exception as e:
    print(f"Không thể tải tự động split='train'. Thử tải toàn bộ dataset...")
    dataset = load_dataset("upwitu/trash_draft_am")
    # Lấy split mặc định nếu có (ví dụ: 'train')
    if isinstance(dataset, dict) or hasattr(dataset, "keys"):
        split_name = list(dataset.keys())[0]
        print(f"Sử dụng split: '{split_name}'")
        dataset = dataset[split_name]

print(f"Dataset loaded. Total samples: {len(dataset)}")
print("Dataset columns (Các cột trong dữ liệu):", dataset.column_names)
print("First sample structure (Cấu trúc dòng đầu tiên):", dataset[0])

# Split dataset into train/evaluation sets (90% train, 10% eval)
if len(dataset) > 0:
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]
    print(f"Dataset split: {len(train_dataset)} train samples, {len(eval_dataset)} evaluation samples")
else:
    train_dataset = dataset
    eval_dataset = dataset

## 4. Formatting and Tokenization

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_sft_data(examples):
    formatted_texts = []
    
    # Trường hợp 1: Dữ liệu chứa cấu trúc DPO/SFT dạng 'prompt' và 'chosen'
    if "prompt" in examples and "chosen" in examples:
        for prompt, chosen in zip(examples["prompt"], examples["chosen"]):
            messages = prompt + chosen
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            formatted_texts.append(text)
            
    # Trường hợp 2: Dữ liệu chứa sẵn danh sách hội thoại 'messages'
    elif "messages" in examples:
        for messages in examples["messages"]:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            formatted_texts.append(text)
            
    # Trường hợp 3: Điều chỉnh tùy biến nếu dataset có cấu trúc khác (ví dụ: cột 'instruction' và 'output')
    elif "instruction" in examples and "output" in examples:
        for inst, out in zip(examples["instruction"], examples["output"]):
            messages = [
                {"role": "user", "content": inst},
                {"role": "assistant", "content": out}
            ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            formatted_texts.append(text)
            
    else:
        raise ValueError(
            f"Cấu trúc dữ liệu lạ với các cột: {list(examples.keys())}. "
            "Hãy cập nhật lại hàm `format_sft_data` để trích xuất hội thoại."
        )
        
    return {
        "text": formatted_texts
    }

if len(train_dataset) > 0:
    formatted_train_dataset = train_dataset.map(format_sft_data, batched=True)
    formatted_eval_dataset = eval_dataset.map(format_sft_data, batched=True) if len(eval_dataset) > 0 else None
    print("Example Formatted SFT text Sample:")
    print(formatted_train_dataset[0]['text'][:500] + "...")
else:
    formatted_train_dataset = train_dataset
    formatted_eval_dataset = eval_dataset
    print("No data available to format.")

## 5. Model Loading and PEFT Configuration

In [ ]:
clear_gpu_memory()

USE_UNSLOTH = True
MAX_SEQ_LENGTH = 2048

if USE_UNSLOTH:
    print("Loading model via Unsloth in BF16...")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,  # Optimized for A100
        load_in_4bit=False,    # Disabled 4-bit quantization since we are on A100
        trust_remote_code=True
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    print("Loading model via Standard HF PEFT in BF16...")
    from transformers import AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,  # Optimized for A100
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

print("Model initialized. Trainable parameters:")
model.print_trainable_parameters()

## 6. SFT Training Configuration and Execution

We configure the SFTTrainer to run validation evaluations at the end of each epoch, load the best model (lowest validation loss) at the end, and only preserve the single best checkpoint to manage disk bounds.

In [ ]:
from trl import SFTTrainer, SFTConfig

resume_checkpoint = None
if os.path.exists(PERSISTENT_DIR):
    checkpoints = [os.path.join(PERSISTENT_DIR, d) for d in os.listdir(PERSISTENT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_checkpoint = checkpoints[-1]
        print(f"Found existing checkpoint: {resume_checkpoint}. Resuming from this step.")

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    output_dir=PERSISTENT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=4,       # Increased batch size for A100
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,                          # A100 performs best on bf16
    bf16=True,                           # True for A100
    optim="paged_adamw_8bit",
    eval_strategy="epoch",               # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save checkpoints at the end of each epoch
    save_total_limit=1,                  # Only save the single best model checkpoint
    load_best_model_at_end=True,         # Load the best model at the end of training
    metric_for_best_model="eval_loss",   # Use validation loss to compare checkpoints
    greater_is_better=False,             # Lower loss is better
    logging_steps=5,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_train_dataset,
    eval_dataset=formatted_eval_dataset,
    tokenizer=tokenizer
)

if len(formatted_train_dataset) > 0:
    print("Starting SFT training...")
    trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
    print("Training completed successfully!")
else:
    print("Dataset is empty.")

## 7. Training Metrics Visualization

We compare the Training Loss against the Validation Loss across steps to identify overfitting.

In [ ]:
import matplotlib.pyplot as plt

if hasattr(trainer, "state") and trainer.state.log_history:
    history = trainer.state.log_history
    
    train_steps = []
    train_losses = []
    eval_steps = []
    eval_losses = []
    
    for log in history:
        step = log.get("step")
        if "loss" in log:
            train_steps.append(step)
            train_losses.append(log["loss"])
        if "eval_loss" in log:
            eval_steps.append(step)
            eval_losses.append(log["eval_loss"])
            
    plt.figure(figsize=(10, 5))
    if train_losses:
        plt.plot(train_steps, train_losses, label="Train Loss", color="red", marker="o")
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="Validation Loss", color="blue", marker="s")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("SFT Loss Comparison: Train vs Validation")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No log history found to plot.")

## 8. Saving and Merging Model Weights

In [ ]:
ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "sft_lora_adapter")
MERGED_PATH = os.path.join(PERSISTENT_DIR, "sft_merged_model")

# Save training adapter checkpoints
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter weights saved to {ADAPTER_PATH}")

# Save merged weights (16-bit precision)
if USE_UNSLOTH:
    print("Saving merged 16-bit model weights via Unsloth...")
    model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to {MERGED_PATH}")
else:
    print("Merging standard HF PEFT model weights...")
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="cpu",
        trust_remote_code=True
    )
    peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    merged_model = peft_model.merge_and_unload()
    
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    print(f"Merged model saved to {MERGED_PATH}")

## 9. Inference Server Deployment

Serve the model locally using vLLM to integrate with the evaluation harness:

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model ./outputs/sft_merged_model \
    --port 8000 \
    --dtype bfloat16 \
    --gpu-memory-utilization 0.90
```

Configure environmental variables in `.env`:
```env
AGENT_LLM=openai/car-bench-llm
OPENAI_API_BASE=http://localhost:8000/v1
OPENAI_API_KEY=local-dummy-key
```